In [ ]:
import sys
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score
import gc

# 1. Add project root to path so we can import 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.loader import GliomaDataLoader
from src.models.cvae import CVAE
from src.utils.config_manager import ConfigManager
from src.notebooks.experiment_utils import setup_experiment

# 2. Setup experiment from config
config_name = 'exp_105'
config, output_dir = setup_experiment(config_name)

# Load paths from config
DATA_PATH = config['paths']['processed_file']
SPLIT_FILE = config['paths']['splits_file']
MODEL_DIR = config['paths']['models_cvae_dir']

print(f"✓ Config loaded: {config_name}")
print(f"Data path: {DATA_PATH}")
print(f"Model weights dir: {MODEL_DIR}")

# 3. Initialize Data Loader FIRST (this applies spectral cropping if defined in config)
print("\nInitializing Loader...")
loader = GliomaDataLoader.from_config(
    h5_data_path=DATA_PATH,
    config=config,
    memory_mapped=False
)

print(f"✓ Loader initialized. Total spectra: {loader.X.shape[0]}")
print(f"✓ Spectral features: {loader.X.shape[1]} (after preprocessing)")

# 4. Build CVAE config from centralized config using actual data shape
cvae_cfg = config['models']['cvae']

# Compute input_shape from actual loaded data (handles cropping automatically)
actual_n_features = loader.X.shape[1]
num_classes_config = config['models']['cnn']['num_classes']

cvae_config = {
    'input_shape': (actual_n_features, 1),
    'num_classes': num_classes_config,
    'latent_dim': cvae_cfg['latent_dim'],
    'loss_weights': {
        'beta': cvae_cfg['loss_weights']['beta'],
        'alpha': cvae_cfg['loss_weights']['alpha']
    },
    'encoder': {
        'filters': cvae_cfg['encoder']['filters'],
        'kernels': cvae_cfg['encoder']['kernels'],
        'strides': cvae_cfg['encoder']['strides'],
        'use_global_pooling': False,
        'use_skip_gap': False
    },
    'decoder': {
        'initial_filters': cvae_cfg['decoder']['initial_filters'],
        'use_dense_projection': True,
        'filters': cvae_cfg['decoder']['filters'],
        'kernels': cvae_cfg['decoder']['kernels'],
        'strides': cvae_cfg['decoder']['strides']
    }
}

print(f"\nCVAE Config loaded from {config_name}:")
print(f"  - Input shape: {cvae_config['input_shape']} (computed from loaded data)")
print(f"  - Latent dim: {cvae_cfg['latent_dim']}")
print(f"  - Beta: {cvae_cfg['loss_weights']['beta']}")
print(f"  - Alpha: {cvae_cfg['loss_weights']['alpha']}")
print(f"  - Num classes: {num_classes_config}")

# Define binary mapping function (only used for exp_105 binary classification)
def map_to_binary(labels):
    """Map 6-class labels to binary: classes [0,1,2] -> 0; classes [3,4,5] -> 1"""
    return (labels >= 3).astype(int)

# Check if this is a binary classification task
is_binary_task = (num_classes_config == 2)
if is_binary_task:
    print(f"  - Binary task: applying 0,1,2→0; 3,4,5→1 mapping")
else:
    print(f"  - Multi-class task: using raw labels")

# 5. Setup Results Dictionary
cbr_results = {
    'Classification_by_Recon': []
}

print("\n🚀 Starting Generative Inference: Classification by Reconstruction...")

# 6. Main Inference Loop
for fold_data in loader:
    fold_name = fold_data['fold_name']
    print(f"\n{'='*60}")
    print(f"🔄 Evaluating {fold_name}")
    print(f"{'='*60}")
    
    # Unpack Real Validation Data
    # Apply binary mapping ONLY for binary classification tasks (exp_105)
    if is_binary_task:
        y_val_compare = map_to_binary(y_val)  # 0,1,2 -> 0; 3,4,5 -> 1
    else:
        y_val_compare = y_val  # Use raw labels for multi-class tasks
        
    X_val_exp = np.expand_dims(X_val, axis=-1) # Shape: (N, actual_n_features, 1)
    n_samples = X_val.shape[0]
    
    # --- MODEL INITIALIZATION ---
    tf.keras.backend.clear_session()
    cvae = CVAE(cvae_config)
    cvae.compile(optimizer='adam')
    
    # Initialize all tensor shapes with a dummy batch
    dummy_x = tf.zeros((2, actual_n_features, 1))
    dummy_y = tf.convert_to_tensor(tf.keras.utils.to_categorical([0, 1], num_classes=num_classes_config), dtype=tf.float32)
    cvae.train_step((dummy_x, dummy_y))
    
    # Load the specific weights for this fold
    enc_path = f"{MODEL_DIR}/encoder_{fold_name}.weights.h5"
    dec_path = f"{MODEL_DIR}/decoder_{fold_name}.weights.h5"
    
    print(f"Loading encoder: {enc_path}")
    print(f"Loading decoder: {dec_path}")
    cvae.encoder.load_weights(enc_path)
    cvae.decoder.load_weights(dec_path)
    print(f"Loading encoder: {enc_path}")
    encoder = cvae.encoder
    decoder = cvae.decoder
    
    # --- PHASE 1: ENCODE ---
    print("  Extracting latent representations (z)...")
    z_mean, _, _, _ = encoder.predict(X_val_exp, batch_size=256, verbose=0)
    
    # --- PHASE 2: DECODE (for all classes) ---
    print(f"  Computing {num_classes_config}-class reconstruction MSE...")
    mse_scores = np.zeros((n_samples, num_classes_config), dtype=np.float32)
    
    for class_idx in range(num_classes_config):
        requested_classes = tf.convert_to_tensor(
            tf.keras.utils.to_categorical([class_idx] * n_samples, num_classes=num_classes_config),
            dtype=tf.float32
        )
        
        # "If this spectrum was Class X, what should it look like?"
        reconstructed_spectra = decoder.predict([z_mean, requested_classes], batch_size=256, verbose=0)
        reconstructed_spectra = np.squeeze(reconstructed_spectra, axis=-1)
        
        # Calculate Mean Squared Error across all spectral bands
        mse = np.mean((X_val - reconstructed_spectra)**2, axis=1)
        mse_scores[:, class_idx] = mse
        
    # --- PHASE 3: CLASSIFY ---
    # Compare against appropriate labels (binary-mapped for exp_105, raw for multi-class)
    acc = accuracy_score(y_val_compare, y_pred_cbr)
    
    # Compare against binary-mapped labels (same mapping as CVAE training)
    acc = accuracy_score(y_val_binary, y_pred_cbr)
    cbr_results['Classification_by_Recon'].append(acc)
    print(f"✅ {fold_name} Classification by Reconstruction Accuracy: {acc:.4f}")
    
    # Cleanup for next fold
    del cvae, encoder, decoder, X_val, y_val, X_val_exp, z_mean, mse_scores
    tf.keras.backend.clear_session()
    gc.collect()

# --- Print Final Summary ---
print("\n" + "="*60)
print("🎉 === CLASSIFICATION BY RECONSTRUCTION COMPLETE ===")
print("="*60)
mean_acc = np.mean(cbr_results['Classification_by_Recon'])
std_acc = np.std(cbr_results['Classification_by_Recon'])

print(f"Classification by Reconstruction: {mean_acc:.4f} ± {std_acc:.4f}")
print("🎉 === CLASSIFICATION BY RECONSTRUCTION COMPLETE ===")

print("="*60)
print("="*60)print("="*60)


mean_acc = np.mean(cbr_results['Classification_by_Recon'])print(f"Classification by Reconstruction: {mean_acc:.4f} ± {std_acc:.4f}")
std_acc = np.std(cbr_results['Classification_by_Recon'])


EXPERIMENT SETUP

Working directory: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data

 Loaded configuration: exp_105
   Description: TSAR Regimen: Fingerprint crop (700-1800), patient cap (500), and dynamic Raman augmentation

 Output directory: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\results
  Experiment already registered (re-running)

--------------------------------------------------------------------------------
Configuration Summary:
--------------------------------------------------------------------------------
  Preprocessing:
    - AirPLS lambda: 100
    - Signal length: 1738
    - Crop region: [700, 1800]

  CV Strategy:
    - Split source: data/splits/strict_splits.json
    - Use group-aware CV: True
    - Hold out test set: True

  Models:
    - RF n_estimators: 100
    - SVM C: 1.0
    - CVAE epochs: 70


✓ Config loaded: exp_105
Data path: results/exp_102/data/glioma_dataset.h5
Model weights dir: results/exp_105/models/cvae

Initializing 

In [2]:
%reset -f

In [ ]:
# DIAGNOSTIC: Check class distributions and verify mapping strategy
print("\n" + "="*60)
print(f"CLASS DISTRIBUTION DIAGNOSTIC (num_classes={num_classes_config})")
print("="*60)

# Check validation labels
fold_iter = iter(loader)
test_fold = next(fold_iter)
X_val_test = test_fold['val'][0].astype(np.float32)
y_val_test = test_fold['val'][1].copy()

if is_binary_task:
    print(f"\n1. Raw validation labels (original 6-class):")
else:
    print(f"\n1. Raw validation labels:") 
print(f"   Unique values: {np.unique(y_val_test)}")
print(f"   Value counts: {np.bincount(y_val_test.astype(int))}")

# Apply 0-indexing if needed
if np.min(y_val_test) == 1:
    y_val_test = y_val_test - 1
    
print(f"\n2. After 0-indexing:")
print(f"   Unique values: {np.unique(y_val_test)}")
print(f"   Value counts: {np.bincount(y_val_test.astype(int))}")

# Apply binary mapping
y_val_test_binary = map_to_binary(y_val_test)
# Apply mapping only if binary task
if is_binary_task:
    y_val_test_compare = map_to_binary(y_val_test)
    print(f"\n3. After binary mapping (0,1,2→0; 3,4,5→1):")
    print(f"   Unique values: {np.unique(y_val_test_compare)}")
    print(f"   Value counts: {np.bincount(y_val_test_compare.astype(int))}")
else:
    y_val_test_compare = y_val_test
    print(f"\n3. Using raw labels (no mapping for multi-class):")
    print(f"   Unique values: {np.unique(y_val_test_compare)}")
    print(f"   Value counts: {np.bincount(y_val_test_compare.astype(int))}")

# Check if binary_mode is being applied
binary_mode = config.get('binary_mode', True)
print(f"\n4. Binary mode setting in config: {binary_mode}")
print(f"   num_classes from config: {config['models']['cnn']['num_classes']}")

# Try a single prediction to see MSE values
X_val_exp = np.expand_dims(X_val_test[:100], axis=-1)

tf.keras.backend.clear_session()
cvae = CVAE(cvae_config)
cvae.compile(optimizer='adam')
dummy_x = tf.zeros((2, actual_n_features, 1))
dummy_y = tf.convert_to_tensor(tf.keras.utils.to_categorical([0, 1], num_classes=num_classes_config), dtype=tf.float32)
cvae.train_step((dummy_x, dummy_y))

enc_path = f"{MODEL_DIR}/encoder_fold_0.weights.h5"
dec_path = f"{MODEL_DIR}/decoder_fold_0.weights.h5"
cvae.encoder.load_weights(enc_path)
cvae.decoder.load_weights(dec_path)

encoder = cvae.encoder
decoder = cvae.decoder

z_mean, _, _, _ = encoder.predict(X_val_exp, batch_size=256, verbose=0)

# Check MSE for both classes on a small sample
mse_sample = np.zeros((100, num_classes_config), dtype=np.float32)
for class_idx in range(num_classes_config):
    requested_classes = tf.convert_to_tensor(
        tf.keras.utils.to_categorical([class_idx] * 100, num_classes=num_classes_config),
        dtype=tf.float32
    )
    reconstructed = decoder.predict([z_mean, requested_classes], batch_size=256, verbose=0)
if is_binary_task:
    print(f"   True labels (6-class):        {y_val_test[:20]}")
    print(f"   True labels (binary mapped):  {y_val_test_compare[:20]}")
else:
    print(f"   True labels:                  {y_val_test_compare[:20]}")
print(f"   Predicted labels:             {y_pred_sample[:20]}")
print(f"   MSE for class 0:  {mse_sample[:5, 0]}")
# Check accuracy against appropriate labels
acc_normal = np.mean(y_pred_sample == y_val_test_compare[:100])
print(f"\n6. Accuracy check:")
print(f"   Normal predictions: {acc_normal:.4f}")

if is_binary_task:
    acc_inverted = np.mean((1 - y_pred_sample) == y_val_test_compare[:100])
    print(f"   Inverted predictions: {acc_inverted:.4f}")
    if acc_inverted > acc_normal * 1.2:
        print(f"   ⚠️  PREDICTIONS MAY BE INVERTED! Inverted accuracy is higher.")
    else:
        print(f"   ✅ Predictions look correct (normal accuracy higher than inverted).")
    print(f"   ⚠️  PREDICTIONS MAY BE INVERTED! Inverted accuracy is higher.")
else:

    print(f"   ⚠️  PREDICTIONS MAY BE INVERTED! Inverted accuracy is higher.")

print("="*60)
else:print("="*60)


    print(f"   ✅ Predictions look correct (normal accuracy higher than inverted).")


BINARY CLASSIFICATION DIAGNOSTIC

 Using CV splits from: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\splits\strict_splits.json
    Found 'cv_folds' section with 5 folds
   ! Separate held-out test set exists: 6 samples
   !  These samples will NOT be used in CV training/validation
   Fold 0: 160084 train samples, 45936 val samples

1. Raw validation labels (original 6-class):
   Unique values: [0 1 2 3 4 5]
   Value counts: [ 2324 14645  3261  9787 12614  3305]

2. After 0-indexing:
   Unique values: [0 1 2 3 4 5]
   Value counts: [ 2324 14645  3261  9787 12614  3305]

3. After binary mapping (0,1,2→0; 3,4,5→1):
   Unique values: [0 1]
   Value counts: [20230 25706]

4. Binary mode setting in config: True
   num_classes from config: 2

5. Sample predictions (first 20 samples):
   True labels (6-class):        [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
   True labels (binary mapped):  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
   Predicted labels (binary):    [0 1 0 0 0 